<a href="https://colab.research.google.com/github/bridgetagboyie16-eng/Photo-to-art-ai-app/blob/main/PhotoToArt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow gradio

In [12]:
import cv2
import numpy as np
from PIL import Image

def extract_lineart(input_image):
    """
    Universal structural extractor: works for all skin tones,
    portraits, landscapes, objects, and animals.
    """
    resized_img = input_image.resize((512, 512))
    img_array = np.array(resized_img)

    # 1. Convert to grayscale
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

    # 2. Normalize brightness so pale, olive, and dark skin tones are equally balanced
    normalized = cv2.equalizeHist(gray)

    # 3. Soft bilateral filter: cleans skin noise while keeping true anatomical edges crisp
    filtered = cv2.bilateralFilter(normalized, d=7, sigmaColor=50, sigmaSpace=50)

    # 4. Adaptive edge detection tailored to the image's own lighting
    v = np.median(filtered)
    lower = int(max(20, (1.0 - 0.33) * v))
    upper = int(min(200, (1.0 + 0.33) * v))
    edges = cv2.Canny(filtered, lower, upper)

    # 5. Expand lines slightly so the model locks onto facial & structural likeness
    kernel = np.ones((2, 2), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)

    edges_3channel = np.stack([edges] * 3, axis=-1)
    return Image.fromarray(edges_3channel)

print("✅ Universal Structure Extractor loaded!")

✅ Universal Structure Extractor loaded!


In [4]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

print("⏳ 1/3 Downloading the Stencil Assistant (ControlNet Canny)...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)

print("⏳ 2/3 Downloading the Master Painter (Stable Diffusion 1.5)...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)

print("⏳ 3/3 Optimizing speed and sending to the GPU...")
# UniPCMultistepScheduler makes the AI paint in 20 crisp steps instead of 50
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Move the whole model onto Google's free T4 GPU
pipe = pipe.to("cuda")

# Low-memory optimization so your session never crashes
pipe.enable_attention_slicing()

print("✅ The AI Brain is fully loaded and ready to paint!")

⏳ 1/3 Downloading the Stencil Assistant (ControlNet Canny)...
⏳ 2/3 Downloading the Master Painter (Stable Diffusion 1.5)...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

⏳ 3/3 Optimizing speed and sending to the GPU...
✅ The AI Brain is fully loaded and ready to paint!


In [13]:
import gradio as gr

# Medium presets focused strictly on authentic traditional art techniques
UNIVERSAL_MEDIUMS = {
    "Graphite Pencil Fine Art": {
        "style": (
            "masterpiece fine graphite pencil drawing, hand-drawn on heavy cold-press sketchbook paper, "
            "realistic pencil shading, cross-hatching, blending stump gradients, 2B and 6B graphite tones, "
            "visible delicate pencil strokes, clean artistic white vignette background, timeless gallery art"
        ),
        "negative": (
            "color, photorealistic camera, glossy, 3d render, digital plastic, heavy black outlines, "
            "deformed, bad eyes, unnatural smooth skin, blur"
        ),
        "control_scale": 1.05
    },
    "Vibrant Watercolor Wash": {
        "style": (
            "delicate watercolor painting, transparent fluid washes, wet-on-wet paint bleeds, "
            "cold press textured watercolor paper, hand-painted fine art, soft edges, vivid pigment pooling"
        ),
        "negative": "greyscale, monochrome, digital vector art, 3d render, plastic, sharp harsh lines",
        "control_scale": 0.95
    },
    "Classical Oil on Canvas": {
        "style": (
            "traditional classical oil painting, rich impasto texture, visible brush work, "
            "chiaroscuro lighting, textured linen canvas, museum masterwork"
        ),
        "negative": "cartoon, 3d render, anime, flat vector, low contrast, washed out",
        "control_scale": 0.90
    },
    "Charcoal & Chalk Study": {
        "style": (
            "expressive charcoal sketch, smudged powdered charcoal, bold gestural strokes, "
            "white chalk highlights, rough grey paper texture, dramatic contrast"
        ),
        "negative": "color, photo, smooth plastic, digital render, sharp cartoon ink",
        "control_scale": 1.0
    },
    "Vintage Pen & Ink": {
        "style": (
            "vintage dip pen and black ink illustration, fine stippling, intricate cross-hatching, "
            "classic bookplate engraving style, clean ivory paper background"
        ),
        "negative": "color, blurry, photograph, soft airbrush, 3d render",
        "control_scale": 1.1
    }
}

def render_art(input_image, selected_medium, optional_subject_desc):
    if input_image is None:
        return None, None

    # 1. Extract structural guide
    edge_map = extract_lineart(input_image)

    # 2. Retrieve the selected artistic medium
    preset = UNIVERSAL_MEDIUMS[selected_medium]

    # If the user leaves the box blank, default to neutral "the subject"
    subject_text = optional_subject_desc.strip() if optional_subject_desc.strip() else "the subject"

    full_prompt = f"a detailed portrait/study of {subject_text}, {preset['style']}"
    full_negative = f"{preset['negative']}, distorted anatomy, blurry features"

    # 3. Generate art
    output = pipe(
        prompt=full_prompt,
        negative_prompt=full_negative,
        image=edge_map,
        num_inference_steps=28,
        guidance_scale=7.5,
        controlnet_conditioning_scale=preset["control_scale"]
    ).images[0]

    return edge_map, output

with gr.Blocks(title="Universal Photo-to-Art Studio") as app:
    gr.Markdown("# 🎨 Universal Photo-to-Art AI Studio")
    gr.Markdown("Convert any photo—portraits of any background, landscapes, pets, or still lifes—into fine traditional artwork.")

    with gr.Row():
        with gr.Column():
            input_box = gr.Image(type="pil", label="Upload Any Photo")

            medium_dropdown = gr.Dropdown(
                choices=list(UNIVERSAL_MEDIUMS.keys()),
                value="Graphite Pencil Fine Art",
                label="Choose Art Medium"
            )

            subject_box = gr.Textbox(
                label="Subject Details (Optional - leave blank or describe in 2-3 words)",
                placeholder="e.g., smiling woman, elderly man, mountain valley, cat sleeping",
                value=""
            )

            generate_btn = gr.Button("✨ Create Artwork", variant="primary")

        with gr.Column():
            stencil_display = gr.Image(type="pil", label="Extracted Structural Guide")
            output_display = gr.Image(type="pil", label="Rendered Art Piece")

    generate_btn.click(
        fn=render_art,
        inputs=[input_box, medium_dropdown, subject_box],
        outputs=[stencil_display, output_display]
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2b998d561915cfeb6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/28 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2b998d561915cfeb6c.gradio.live


In [14]:
!pip install -q timm transformers

In [15]:
import numpy as np
from PIL import Image
from transformers import pipeline as hf_pipeline

print("⏳ Loading DPT 3D Depth Estimator...")
# Intel MiDaS depth model predicts relative 3D spatial distance for every pixel
depth_estimator = hf_pipeline("depth-estimation", model="Intel/dpt-hybrid-midas")

def extract_depth_map(input_image):
    """
    Extracts a dense 3D topological map from any image (portraits, landscapes, objects).
    Preserves exact facial proportions, nose bridge, jawline, and depth planes.
    """
    # Resize to standard generation resolution
    resized_image = input_image.resize((512, 512))

    # Run depth estimation
    depth_output = depth_estimator(resized_image)["depth"]

    # Normalize depth values across the 0-255 grayscale range
    depth_array = np.array(depth_output)
    depth_normalized = (
        (depth_array - depth_array.min()) / (depth_array.max() - depth_array.min()) * 255.0
    ).astype("uint8")

    # Format into a 3-channel RGB image for ControlNet
    depth_3ch = np.stack([depth_normalized] * 3, axis=-1)
    return Image.fromarray(depth_3ch)

print("✅ 3D Depth Extractor ready!")

⏳ Loading DPT 3D Depth Estimator...


config.json:   0%|          | 0.00/9.88k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  490MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/414 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  490MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ 3D Depth Extractor ready!


In [16]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

print("⏳ 1/2 Loading ControlNet Depth model...")
controlnet_depth = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

print("⏳ 2/2 Assembling Stable Diffusion 1.5 + Depth Pipeline...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet_depth,
    torch_dtype=torch.float16
)

# Use UniPC scheduler for clean outputs in ~20-25 steps
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Send pipeline to the T4 GPU and enable memory-saving slicing
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

print("✅ Depth-conditioned generative pipeline loaded and ready!")

⏳ 1/2 Loading ControlNet Depth model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.45GB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

...

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

✅ Depth-conditioned generative pipeline loaded and ready!


In [ ]:
import gradio as gr

# Presets engineered for authentic physical art textures
MEDIUM_PRESETS = {
    "Graphite Pencil Portrait": {
        "style": (
            "masterpiece fine graphite pencil drawing, authentic portrait study, "
            "delicate pencil shading, cross-hatching, blending stump gradients, "
            "2B and 6B graphite tones, clean textured cold-press fine art paper, hand-drawn fine art"
        ),
        "negative": (
            "color, photorealistic glossy skin, 3d digital render, smooth plastic, cartoon, "
            "deformed, altered facial structure, blurry, harsh black borders"
        ),
        "control_scale": 1.15
    },
    "Vibrant Watercolor": {
        "style": (
            "traditional watercolor painting, delicate fluid washes, wet-on-wet paint bleeding, "
            "cold press textured watercolor paper, hand-painted fine art, soft artistic edges"
        ),
        "negative": "greyscale, monochrome, digital vector art, 3d render, plastic, sharp harsh digital lines",
        "control_scale": 1.0
    },
    "Classical Oil on Canvas": {
        "style": (
            "traditional classical oil painting, rich impasto texture, visible expressive brushstrokes, "
            "chiaroscuro lighting, textured linen canvas, museum quality masterwork"
        ),
        "negative": "cartoon, 3d render, anime, flat vector, low contrast, washed out digital",
        "control_scale": 0.95
    },
    "Expressive Charcoal Study": {
        "style": (
            "expressive charcoal portrait sketch, smudged powdered charcoal, bold gestural strokes, "
            "white chalk highlights, rough grey paper texture, dramatic gallery contrast"
        ),
        "negative": "color, photograph, smooth plastic, digital render, sharp vector",
        "control_scale": 1.1
    }
}

def render_artwork(input_image, selected_medium):
    if input_image is None:
        return None, None

    # 1. Extract 3D topographic geometry
    depth_map = extract_depth_map(input_image)

    # 2. Get prompt configuration
    preset = MEDIUM_PRESETS[selected_medium]
    full_prompt = f"a detailed artistic depiction of the subject, {preset['style']}"
    full_negative = f"{preset['negative']}, distorted anatomy, bad eyes, disfigured"

    # 3. Generate artwork conditioned on 3D depth
    output = pipe(
        prompt=full_prompt,
        negative_prompt=full_negative,
        image=depth_map,
        num_inference_steps=26,
        guidance_scale=8.0,
        controlnet_conditioning_scale=preset["control_scale"]
    ).images[0]

    return depth_map, output

# Build web interface
with gr.Blocks(title="Photo-to-Art AI Studio") as app:
    gr.Markdown("# 🎨 Universal Photo-to-Art Studio")
    gr.Markdown("Transform any photo into traditional fine art mediums while preserving true 3D facial likeness and structure.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Upload Photo (Portrait, Landscape, or Object)")
            medium_dropdown = gr.Dropdown(
                choices=list(MEDIUM_PRESETS.keys()),
                value="Graphite Pencil Portrait",
                label="Select Art Medium"
            )
            generate_btn = gr.Button("✨ Transform into Art", variant="primary")

        with gr.Column():
            depth_preview = gr.Image(type="pil", label="1. 3D Facial Depth Topography")
            output_preview = gr.Image(type="pil", label="2. Rendered Artwork")

    generate_btn.click(
        fn=render_artwork,
        inputs=[input_img, medium_dropdown],
        outputs=[depth_preview, output_preview]
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0981148fe25344a5f1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
